In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
import pickle

import pandas as pd
import category_encoders as ce
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder
from common import correlation_based_feature_selection as cbfs

In [ ]:
# Read in data
molecular_features = pd.read_pickle("molecular_features.pkl")
print(molecular_features.shape)

# 1. Input Features

## 1.1 Molecular Features

In [ ]:
X_molecular = molecular_features.copy()
X_molecular = X_molecular.drop(columns=["Progression Free Survival", "Event"])

# split subjects into Discovery and Replicate cohorts
discovery_molecular = X_molecular[X_molecular["Cohort"] == "Discovery"].copy()
replicate_molecular = X_molecular[X_molecular["Cohort"] == "Replicate"].copy()

def categorize_molecular_subtype(subtype):
    s = str(subtype).upper()
    if "KIAA1549" in s: return "KIAA1549_BRAF"
    if "V600E"    in s: return "BRAF_V600E"
    if "NF1"      in s: return "NF1"
    if "FGFR"     in s: return "FGFR"
    if "RTK"      in s: return "RTK"
    if "IDH"      in s: return "IDH"
    if "MYB"      in s: return "MYB"
    if "MAPK"     in s: return "other_MAPK"
    if "WILDTYPE" in s: return "wildtype"
    return "other"

discovery_molecular["mol_group"] = discovery_molecular["Molecular Subtype"].apply(categorize_molecular_subtype)
replicate_molecular["mol_group"] = replicate_molecular["Molecular Subtype"].apply(categorize_molecular_subtype)


# Check distributions
print("\nDiscovery:")
print(discovery_molecular["mol_group"].value_counts())
print("\nReplicate:")
print(replicate_molecular["mol_group"].value_counts())

# ── Save mapping for verification ────────────────────────────────────────
mapping = pd.concat([
    discovery_molecular[["Molecular Subtype", "mol_group"]].assign(Cohort="Discovery"),
    replicate_molecular[["Molecular Subtype", "mol_group"]].assign(Cohort="Replicate"),
])

# Unique subtype → group mapping with counts
mapping_summary = (
    mapping.groupby(["Molecular Subtype", "mol_group", "Cohort"])
    .size()
    .reset_index(name="n")
    .sort_values(["mol_group", "n"], ascending=[True, False])
)

mapping_summary.to_csv("./molecular_subtype_mapping.csv", index=False)
print(mapping_summary.to_string(index=False))

# One-hot encode — fit on discovery only, wildtype as reference
categories = ["KIAA1549_BRAF", "BRAF_V600E", "NF1", "FGFR",
              "RTK", "IDH", "MYB", "other_MAPK", "other", "wildtype"]

encoder = OneHotEncoder(
    categories=[categories],
    sparse_output=False,
    drop=["wildtype"],
    handle_unknown="ignore"
)

encoder.fit(discovery_molecular[["mol_group"]])

disc_encoded = encoder.transform(discovery_molecular[["mol_group"]])
rep_encoded  = encoder.transform(replicate_molecular[["mol_group"]])

cols = encoder.get_feature_names_out(["mol_group"])

discovery_molecular[cols] = pd.DataFrame(disc_encoded, index=discovery_molecular.index)
replicate_molecular[cols] = pd.DataFrame(rep_encoded,  index=replicate_molecular.index)

# Save encoder
with open("./molecular_subtype_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

# Drop working columns
discovery_molecular = discovery_molecular.drop(columns=["Molecular Subtype", "mol_group"])
replicate_molecular = replicate_molecular.drop(columns=["Molecular Subtype", "mol_group"])

# Concatenate cohorts
X_molecular = pd.concat([discovery_molecular, replicate_molecular])

# Cache
X_molecular.to_pickle("X_molecular.pkl")
print(f"\nFinal shape: {X_molecular.shape}")
print(f"Features: {encoder.get_feature_names_out(['mol_group']).tolist()}")

In [ ]:
X_molecular

# 2. Output Features

In [ ]:
# select columns
y = molecular_features[["Progression Free Survival", "Event"]].merge(
    X_molecular["Cohort"].to_frame(), left_index=True, right_index=True
)
# compute age in months
y["Progression Free Survival"] = y["Progression Free Survival"].apply(
    lambda x: int(x) / 30.417
)
# cache output features
y.to_pickle("./y.pkl")
y.shape

In [ ]:
y